# Lesson 4: Multimodal messages

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [3]:
print(uploader.value)

({'name': 'prost.png', 'type': 'image/png', 'size': 2319987, 'content': <memory at 0x000001DDFE4084C0>, 'last_modified': datetime.datetime(2026, 8, 23, 16, 29, 43, 741000, tzinfo=datetime.timezone.utc)},)


In [4]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [6]:
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this image"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

model = init_chat_model(
    model="models/gemini-3.5-flash", 
    model_provider="google_genai"
)

agent = create_agent(
    model=model,
)

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': "This image captures a classic piece of Formula 1 history from the late 1990s. Here are the details about the car, the driver, and the context of the photo:\n\n### 1. The Car and Team\n* **Team:** **Prost Grand Prix** (Prost Peugeot). The team was owned and run by the legendary four-time F1 World Champion, Alain Prost, who bought the Ligier team in 1997 and rebranded it.\n* **Model:** This is the **Prost AP02**, which competed in the **1999 Formula One World Championship**. \n* **Engine:** The car was powered by a 3.0-liter Peugeot V10 engine.\n\n### 2. The Driver\n* Driven by French driver **Olivier Panis** (his name is visible on the side of the cockpit/airbox next to the French flag, and his distinct blue and white helmet is visible). Panis is best known for his dramatic victory at the wet 1996 Monaco Grand Prix. 1999 was his final year driving for the Prost team.\n\n### 3. Livery and Sponsorships\nThe car features a striking dark French blue livery, which 

In [7]:
print(response['messages'][-1].content[0]['text'])

This image captures a classic piece of Formula 1 history from the late 1990s. Here are the details about the car, the driver, and the context of the photo:

### 1. The Car and Team
* **Team:** **Prost Grand Prix** (Prost Peugeot). The team was owned and run by the legendary four-time F1 World Champion, Alain Prost, who bought the Ligier team in 1997 and rebranded it.
* **Model:** This is the **Prost AP02**, which competed in the **1999 Formula One World Championship**. 
* **Engine:** The car was powered by a 3.0-liter Peugeot V10 engine.

### 2. The Driver
* Driven by French driver **Olivier Panis** (his name is visible on the side of the cockpit/airbox next to the French flag, and his distinct blue and white helmet is visible). Panis is best known for his dramatic victory at the wet 1996 Monaco Grand Prix. 1999 was his final year driving for the Prost team.

### 3. Livery and Sponsorships
The car features a striking dark French blue livery, which was highly recognizable during this er

# 2. Audio input

In [9]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.84it/s]

Done.


In [10]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': "Based on your request, here is a detailed overview of the popular Pokémon, **Blaziken**:\n\n### **Overview**\n* **Name:** Blaziken (Japanese: Bursyamo)\n* **Classification:** The Blaze Pokémon\n* **Type:** Fire / Fighting\n* **Generation:** Generation III (Hoenn region)\n* **National Dex Number:** #0257\n\n---\n\n### **Evolutionary Line**\nBlaziken is the final evolution of **Torchic**, the Fire-type starter Pokémon of the Hoenn region. \n1. **Torchic** evolves into **Combusken** starting at Level 16.\n2. **Combusken** evolves into **Blaziken** starting at Level 36.\n\n---\n\n### **Appearance and Design**\nBlaziken is a bipedal, chicken-like Pokémon. \n* It is predominantly red with beige accents on its chest and legs, and yellow highlights.\n* It features long, white, feather-like hair extending from the back of its head.\n* It has strong, muscular legs designed for high jumping and powerful kicks.\n* When in battle, it can release flames from its wrists, wr

In [11]:
print(response['messages'][-1].content[0]['text'])

Based on your request, here is a detailed overview of the popular Pokémon, **Blaziken**:

### **Overview**
* **Name:** Blaziken (Japanese: Bursyamo)
* **Classification:** The Blaze Pokémon
* **Type:** Fire / Fighting
* **Generation:** Generation III (Hoenn region)
* **National Dex Number:** #0257

---

### **Evolutionary Line**
Blaziken is the final evolution of **Torchic**, the Fire-type starter Pokémon of the Hoenn region. 
1. **Torchic** evolves into **Combusken** starting at Level 16.
2. **Combusken** evolves into **Blaziken** starting at Level 36.

---

### **Appearance and Design**
Blaziken is a bipedal, chicken-like Pokémon. 
* It is predominantly red with beige accents on its chest and legs, and yellow highlights.
* It features long, white, feather-like hair extending from the back of its head.
* It has strong, muscular legs designed for high jumping and powerful kicks.
* When in battle, it can release flames from its wrists, wrapping its hands in fire to punch opponents.

---
